# Course Navigator & RAG Assistant

Welcome to the **Course Navigator & RAG Assistant** notebook! This tool lets you semantically search and ask questions across all the guides, weekly lectures, and source code of the **LLM Engineering** repository.

This notebook demonstrates how the RAG system works programmatically under the hood.

### 1. Load Environment and Imports

We need to import the required libraries and add the parent directory to the Python path to import our `indexer` and `searcher` modules.

In [ ]:
import sys
from pathlib import Path

# Ensure the current directory is in the path
current_dir = Path.cwd()
if str(current_dir) not in sys.path:
    sys.path.append(str(current_dir))

# Verify files exist
print("Indexer file exists:", (current_dir / "indexer.py").exists())
print("Searcher file exists:", (current_dir / "searcher.py").exists())

### 2. Run Indexer (Optional)

If you haven't built the index yet, or if you've made updates to the course notebooks, python files, or guides, uncomment and run the cell below to build the index database `navigator_index.pkl`.

In [ ]:
# import indexer
# indexer.build_index()

### 3. Initialize the Course Searcher

We will load our `CourseSearcher` class, which automatically loads the `all-MiniLM-L6-v2` embedding model and the pickled search index.

In [ ]:
from searcher import CourseSearcher

searcher = CourseSearcher()

### 4. Run Semantic Search

Let's test semantic retrieval. When we run a query, the searcher embeds it and uses cosine similarity to rank all indexed chunks of the repository. We will retrieve the top 3 matches and inspect their metadata and content.

In [ ]:
query = "How does prompt engineering and tool calling work with Anthropic Claude models?"
results = searcher.search(query, top_k=3)

print(f"--- Top Semantic Matches for: '{query}' ---\n")
for i, res in enumerate(results, 1):
    meta = res["chunk"]["metadata"]
    score = res["score"]
    content = res["chunk"]["content"]
    
    print(f"{i}. Match Score: {score:.4f}")
    print(f"   File Path:   {meta.get('file_path')}")
    print(f"   File Type:   {meta.get('file_type')}")
    print(f"   Snippet:\n{content[:200]}...")
    print("-" * 50)

### 5. Chat with the Assistant (RAG)

Now we'll send the query and retrieved context to the LLM (using the OpenRouter API credentials specified in your `.env` file). The assistant will construct a synthesized answer and reference the files where it found the information.

In [ ]:
answer = searcher.answer_question(query, top_k=4)
print("=== AI COURSE NAVIGATOR ANSWER ===")
print(answer)

### 6. Running the Web UI

To launch a beautiful interactive web interface to search and chat, you can run the `app.py` script from your command line:
```bash
python app.py
```
This will launch a Gradio interface at http://127.0.0.1:7860/ where you can chat and explore matches.